# 02 Projection and Clustering

This notebook takes the feature table from Notebook 01, standardizes the descriptors, projects them into 3D, and groups similar frames with clustering.

It is a good place to compare PCA, UMAP, and t-SNE on your own machine.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN, KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

try:
    import umap
    HAS_UMAP = True
except Exception:
    HAS_UMAP = False

plt.rcParams['figure.figsize'] = (10, 7)
plt.rcParams['axes.grid'] = False

In [ ]:
INPUT_CSV = Path('analysis_output/features.csv')
OUTPUT_DIR = Path('analysis_output')
OUTPUT_DIR.mkdir(exist_ok=True)

PROJECTION_METHOD = 'pca'  # 'pca', 'umap', or 'tsne'
N_CLUSTERS = 8
RANDOM_STATE = 42

print(f'Input CSV: {INPUT_CSV.resolve()}')
print(f'UMAP available: {HAS_UMAP}')

In [ ]:
def load_features(csv_path):
    frame = pd.read_csv(csv_path)
    return frame

def select_feature_columns(frame):
    excluded = {'frame_index', 'time'}
    return [column for column in frame.columns if column not in excluded]

def project_features(frame, method='pca'):
    feature_columns = select_feature_columns(frame)
    values = frame[feature_columns].fillna(0.0).to_numpy()
    values = StandardScaler().fit_transform(values)

    if method == 'umap' and HAS_UMAP:
        reducer = umap.UMAP(n_components=3, n_neighbors=25, min_dist=0.05, metric='euclidean', random_state=RANDOM_STATE)
        projection = reducer.fit_transform(values)
    elif method == 'tsne':
        perplexity = min(30, max(5, len(frame) // 10))
        reducer = TSNE(n_components=3, perplexity=perplexity, init='pca', learning_rate='auto', random_state=RANDOM_STATE)
        projection = reducer.fit_transform(values)
    else:
        reducer = PCA(n_components=3, random_state=RANDOM_STATE)
        projection = reducer.fit_transform(values)

    projected = frame.copy()
    projected['x'] = projection[:, 0]
    projected['y'] = projection[:, 1]
    projected['z'] = projection[:, 2]
    return projected, reducer

def cluster_frames(projected_frame):
    feature_columns = [column for column in projected_frame.columns if column.startswith('mfcc_')] + ['rms', 'centroid', 'flatness', 'rolloff', 'bandwidth', 'zcr']
    values = StandardScaler().fit_transform(projected_frame[feature_columns].fillna(0.0).to_numpy())
    kmeans = KMeans(n_clusters=N_CLUSTERS, n_init='auto', random_state=RANDOM_STATE)
    projected_frame['cluster'] = kmeans.fit_predict(values)
    projected_frame['noise_label'] = DBSCAN(eps=1.6, min_samples=8).fit_predict(values)
    return projected_frame, kmeans

In [ ]:
if INPUT_CSV.exists():
    features = load_features(INPUT_CSV)
    projected, reducer = project_features(features, method=PROJECTION_METHOD)
    clustered, kmeans = cluster_frames(projected)
    display(clustered.head())
    print(f'Projected frames: {len(clustered)}')
    print(f'Method: {PROJECTION_METHOD}')
else:
    print('Run Notebook 01 first or update INPUT_CSV to point at a saved features.csv file.')

In [ ]:
def plot_projection(frame):
    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')
    scatter = ax.scatter(frame['x'], frame['y'], frame['z'], c=frame['cluster'], cmap='turbo', s=18, alpha=0.9)
    ax.set_title(f'{PROJECTION_METHOD.upper()} projection with clusters')
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    fig.colorbar(scatter, ax=ax, label='Cluster')
    plt.show()

if 'clustered' in globals():
    plot_projection(clustered)

In [ ]:
def save_projection(frame, output_dir=OUTPUT_DIR, stem='projection_clusters'):
    csv_path = output_dir / f'{stem}.csv'
    json_path = output_dir / f'{stem}.json'
    frame.to_csv(csv_path, index=False)
    frame.to_json(json_path, orient='records', indent=2)
    return csv_path, json_path

if 'clustered' in globals():
    csv_path, json_path = save_projection(clustered)
    print(f'Saved: {csv_path}')
    print(f'Saved: {json_path}')